# Tags Pipeline v3 — Resolution notes → Tags, Tag 1, Tag 2

Alur: **baca input (Excel/CSV)** → **ambil tags (semua blok, unik)** → **filter `#`** → **logic certificate** → **buang IS High Risk** → **Tag 1 & Tag 2** → **simpan Excel**.

Semua pencocokan **tidak peduli huruf besar/kecil**.

**Yang baru di v3**
1. Kalau ada beberapa `Tags:`, **semua** tag dari semua blok diambil lalu dibuat **unik** (urutan kemunculan pertama dipertahankan). Di v2: hanya blok dengan tag terbanyak.
2. **Filter `#`** — dicek pada gabungan tag dari semua blok:
   - Sebagian tag pakai `#` → yang diambil **hanya** tag yang pakai `#`.
   - Semua tag pakai `#` → semua diambil.
   - Semua tag tidak pakai `#` → semua diambil.
   - Tag yang sama muncul dengan dan tanpa `#` (mis. `Infra` dan `#Infra`) → dihitung sebagai tag ber-`#`.
   - Tanda `#` tetap dibuang dari hasil akhir. Kolom `Tags Raw` menyimpan gabungan tag lengkap dengan `#`-nya, untuk pengecekan.
   - Bisa dimatikan lewat `HASH_FILTER = False`.

**Logic certificate** (dicek setelah filter `#`)
1. Kalau ada tag yang isinya **persis** `Certificate` (atau `Certificates`) → tags = `Certificate`.
   Tag dengan awalan/akhiran lain, mis. `infra_Certificate` atau `Certificate_xyz`, **tidak dihitung** dan diproses sebagai tag biasa.
2. Kalau Resolution notes punya tags (tanpa certificate) → pakai tags itu + logic biasa.
3. Kalau Resolution notes **tidak punya tags** → cari kata `Certificate` **utuh** di Short description & Resolution notes. Ketemu → tags = `Certificate`.
   Kata yang menempel dengan `_` atau `-` (mis. `infra_certificate`) tidak dihitung.

**Tag 1 & Tag 2** (sama seperti v2)
- Tag 1 prioritas: `ticket_duplicated` → `cm_bug` → `cm_`
- Tag 2 prioritas: `attachment` → `on-call` → `FEEDBACK_COMPLIMENT` → `user_request` → `user_`
- Ada prefix Tag 1 → Tag 1 = yang terkuat; Tag 2 = prefix Tag 2 terkuat, kalau tidak ada = tag lain paling awal.
- **Tidak ada** prefix Tag 1 dan ada **≥ 2** prefix Tag 2 → Tag 2 = terkuat ke-1, Tag 1 = terkuat ke-2.
- Tidak ada prefix Tag 1 dan hanya 1 prefix Tag 2 → Tag 2 = itu, Tag 1 = tag lain paling awal.
- Tidak ada prefix sama sekali → Tag 1 & Tag 2 = dua tag pertama sesuai urutan.
- Hanya 1 tag → masuk Tag 1, Tag 2 kosong.

## 1. Pengaturan (ubah di sini saja)

In [ ]:
# ---------- file ----------
INPUT_FILE  = "input.xlsx"          # bisa .xlsx / .xls / .csv
SHEET_NAME  = 0                     # khusus Excel: 0 = sheet pertama, atau nama sheet
OUTPUT_FILE = "output_tags.xlsx"

# ---------- nama kolom input ----------
RESOLUTION_COL = "Resolution notes"
SHORT_DESC_COL = "Short description"

# ---------- nama kolom output ----------
TAGS_RAW_COL  = "Tags Raw"            # gabungan tag semua blok, '#' dipertahankan (untuk cek)
TAGS_COL      = "Tags Extracted"
NO_RISK_COL   = "remove_Is_High_Risk"
SOURCE_COL    = "Tags Source"
TAG1_COL      = "Tag 1"
TAG2_COL      = "Tag 2"

# kolom yang dipakai untuk menentukan Tag 1 & Tag 2
# ganti ke NO_RISK_COL kalau "IS High Risk" tidak boleh ikut jadi Tag 1/Tag 2
TAG12_SOURCE_COL = TAGS_COL

# ---------- filter '#' ----------
HASH_FILTER = True                  # False -> abaikan logic '#', semua tag diambil

# ---------- prefix (urut dari yang paling kuat) ----------
TAG1_PREFIXES = ["ticket_duplicated", "cm_bug", "cm_"]
TAG2_PREFIXES = ["attachment", "on-call", "FEEDBACK_COMPLIMENT", "user_request", "user_"]

# ---------- certificate & IS High Risk ----------
CERT_WORDS   = ["certificate", "certificates"]   # tag harus PERSIS salah satu ini (tanpa awalan/akhiran)
CERT_TAG     = "Certificate"        # isi tags kalau certificate terdeteksi
HIGH_RISK    = "IS High Risk"

## 2. Fungsi: ambil tags dari Resolution notes (semua blok + filter `#`)

In [ ]:
import re
from pathlib import Path
import pandas as pd

BR_RE   = re.compile(r"<\s*br\s*/?\s*>", re.I)
HTML_RE = re.compile(r"</?\s*(?:p|div|span|li|ul|ol|b|i|u|strong|em)\s*/?\s*>", re.I)
TAGS_RE = re.compile(r"tags\s*:", re.I)
CTRL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def clean_text(value):
    """<br/> jadi baris baru, buang HTML ringan & karakter aneh."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    s = BR_RE.sub("\n", str(value))
    s = HTML_RE.sub("\n", s)
    s = s.replace("&nbsp;", " ").replace("\r\n", "\n").replace("\r", "\n")
    return CTRL_RE.sub("", s)


def parse_block(block):
    """Satu blok setelah 'Tags:' -> list (tag, pakai_hash). Berhenti di baris yang mengandung ':'."""
    items = []
    for line in block.split("\n"):
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            break
        for item in line.split(","):
            raw = item.strip()
            tag = raw.lstrip("#").strip()
            if tag:
                items.append((tag, raw.startswith("#")))
    return items


def extract_tag_items(text):
    """
    Gabungkan tag dari SEMUA blok 'Tags:' lalu dibuat unik (urutan kemunculan pertama).
    Tag yang sama muncul dengan & tanpa '#' -> dianggap pakai '#'.
    Hasil: list (tag, pakai_hash).
    """
    merged = {}                                   # key casefold -> [tag, pakai_hash]
    for block in TAGS_RE.split(clean_text(text))[1:]:
        for tag, has_hash in parse_block(block):
            key = tag.casefold()
            if key in merged:
                merged[key][1] = merged[key][1] or has_hash
            else:
                merged[key] = [tag, has_hash]
    return [(tag, has_hash) for tag, has_hash in merged.values()]


def apply_hash_rule(items):
    """
    Sebagian pakai '#'          -> ambil yang pakai '#' saja.
    Semua pakai / semua tidak   -> ambil semua.
    """
    hashed = [t for t, h in items if h]
    if HASH_FILTER and hashed:
        return hashed
    return [t for t, _ in items]

## 3. Fungsi: logic certificate

In [ ]:
def is_cert_tag(tag):
    """True hanya kalau tag PERSIS 'Certificate'/'Certificates'. 'infra_Certificate' -> False."""
    return tag.strip().casefold() in {w.casefold() for w in CERT_WORDS}


def has_cert_word(text):
    """Cari kata certificate UTUH di teks. 'SSL Certificate renewal' -> True, 'infra_certificate' -> False."""
    words = "|".join(re.escape(w) for w in sorted(CERT_WORDS, key=len, reverse=True))
    return re.search(rf"(?<![\w-])(?:{words})(?![\w-])", text, re.I) is not None


def resolve_tags(resolution_note, short_desc):
    """Kembalikan (list tags final, tags raw untuk cek, sumber tags)."""
    items = extract_tag_items(resolution_note)
    raw   = ", ".join(("#" if h else "") + t for t, h in items)
    tags  = apply_hash_rule(items)

    # 1 & 2: Resolution notes punya tags
    if tags:
        if any(is_cert_tag(t) for t in tags):
            return [CERT_TAG], raw, "certificate (dari tags)"
        source = "resolution notes (hanya #)" if len(tags) < len(items) else "resolution notes"
        return tags, raw, source

    # 3: tidak punya tags -> cari kata certificate di short description & resolution notes
    text = clean_text(short_desc) + " " + clean_text(resolution_note)
    if has_cert_word(text):
        return [CERT_TAG], raw, "certificate (dari teks)"

    return [], raw, "tidak ada tags"

## 4. Fungsi: Tag 1 & Tag 2

In [ ]:
def split_tag_string(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return []
    return [t.strip() for t in str(s).split(",") if t.strip()]


def prefix_rank(tag, prefixes):
    """Posisi prefix yang cocok (0 = terkuat). None kalau tidak cocok."""
    t = tag.casefold()
    for rank, p in enumerate(prefixes):
        if t.startswith(p.casefold()):
            return rank
    return None


def strongest(tags, prefixes):
    """Index tag yang cocok prefix, diurutkan dari yang terkuat (lalu urutan muncul)."""
    found = [(prefix_rank(t, prefixes), i) for i, t in enumerate(tags)]
    return [i for rank, i in sorted(f for f in found if f[0] is not None)]


def pick_tag1_tag2(tags):
    n = len(tags)
    if n == 0:
        return "", ""
    if n == 1:
        return tags[0], ""

    t1 = strongest(tags, TAG1_PREFIXES)
    t2 = strongest(tags, TAG2_PREFIXES)
    first_other = lambda skip: next(i for i in range(n) if i != skip)

    if t1:                          # ada prefix Tag 1
        i1 = t1[0]
        i2 = t2[0] if t2 else first_other(i1)
    elif len(t2) >= 2:              # tidak ada Tag 1, ada >= 2 prefix Tag 2
        i2, i1 = t2[0], t2[1]
    elif len(t2) == 1:              # tidak ada Tag 1, cuma 1 prefix Tag 2
        i2 = t2[0]
        i1 = first_other(i2)
    else:                           # tidak ada prefix sama sekali
        i1, i2 = 0, 1

    return tags[i1], tags[i2]

## 5. Cek cepat logic (tidak butuh file input)

Pakai pengaturan default di bagian 1. Kalau pengaturan diubah, sebagian test bisa ikut berubah hasilnya.

In [ ]:
# (resolution notes, short description, Tags Extracted, Tag 1, Tag 2)
CASES = [
    # --- contoh lama ---
    ("Tags:\nTags:\n#User_Request\n", "",
     "User_Request", "User_Request", ""),
    ("Tags:<br/>#CM_BUG_MessageDuplicated, IS High Risk\nTags:\n#CM_BUG_MessageDuplicated, IS High Risk\n"
     "tracker id: 112952 - TRACKER INEXTRACK PMI - Duplicated submission [IS High Risk]\n", "",
     "CM_BUG_MessageDuplicated", "CM_BUG_MessageDuplicated", ""),
    ("Tags:<br/>Tags:\nTags:\nIS High Risk, #DTS_Transmission_Failing, #CM_BUG_MessageDuplicated_31/32\n"
     "tracker id: 100312 - TRACKER INEXTRACK PMI - Message duplicated [IS High Risk]", "",
     "DTS_Transmission_Failing, CM_BUG_MessageDuplicated_31/32",
     "CM_BUG_MessageDuplicated_31/32", "DTS_Transmission_Failing"),
    ("Val version is aligned with PRD<br/>Tags:<br/>#Infra<br/>Tracker ID:\nTags:", "",
     "Infra", "Infra", ""),
    # --- baru: semua blok digabung + unik ---
    ("Tags:\n#CM_item, #User_request\nTags:\n#Attachment_missing, #CM_item", "",
     "CM_item, User_request, Attachment_missing", "CM_item", "Attachment_missing"),
    # --- baru: semua tanpa '#' -> semua diambil ---
    ("Tags:\nUser_request, hyte", "",
     "User_request, hyte", "hyte", "User_request"),
    # --- baru: tag sama muncul dengan & tanpa '#' -> dihitung ber-'#' ---
    ("Tags:\nInfra, #Bug\nTags:\n#Infra", "",
     "Infra, Bug", "Infra", "Bug"),
    # --- certificate: hanya tag yang PERSIS 'Certificate' ---
    ("Tags:\n#Certificate, #CM_x", "",
     "Certificate", "Certificate", ""),
    ("Tags:\n#certificates, #Infra", "",
     "Certificate", "Certificate", ""),
    ("Tags:\n#Certificates_expired, #CM_x", "",                      # ada akhiran -> tag biasa
     "Certificates_expired, CM_x", "CM_x", "Certificates_expired"),
    ("Tags:\n#infra_Certificate, #User_request", "",                 # ada awalan -> tag biasa
     "infra_Certificate, User_request", "infra_Certificate", "User_request"),
    # --- certificate dari teks (tidak ada tags): hanya kata utuh ---
    ("renewed the cert", "SSL Certificate renewal",
     "Certificate", "Certificate", ""),
    ("fix infra_certificate config", "update Certificate_store path",
     "", "", ""),
    # --- tidak ada tag ---
    ("tanpa tag apa pun", "", "", "", ""),
]


def run_selftest():
    failed = 0
    for i, (res, short, exp_tags, exp1, exp2) in enumerate(CASES, 1):
        tags, raw, src = resolve_tags(res, short)
        t1, t2 = pick_tag1_tag2(tags)
        got, expected = (", ".join(tags), t1, t2), (exp_tags, exp1, exp2)
        ok = got == expected
        failed += not ok
        print(f"[{'OK ' if ok else 'FAIL'}] case {i:>2} | raw: {raw!r}")
        print(f"            tags: {got[0]!r} | Tag 1: {t1!r} | Tag 2: {t2!r} | sumber: {src}")
        if not ok:
            print(f"        expected: {expected}")
    print("\n" + ("SEMUA TEST LULUS" if not failed else f"{failed} test GAGAL"))


run_selftest()

## 6. Baca file input

In [ ]:
def find_col(df, name):
    """Cari nama kolom tanpa peduli huruf besar/kecil & spasi."""
    lookup = {str(c).strip().casefold(): c for c in df.columns}
    return lookup.get(name.strip().casefold())


if Path(INPUT_FILE).suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_FILE, dtype=object, sep=None, engine="python", encoding="utf-8-sig")
else:
    df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, dtype=object)

res_col   = find_col(df, RESOLUTION_COL)
short_col = find_col(df, SHORT_DESC_COL)

if res_col is None:
    raise KeyError(f"Kolom '{RESOLUTION_COL}' tidak ada. Kolom tersedia: {list(df.columns)}")
if short_col is None:
    print(f"Peringatan: kolom '{SHORT_DESC_COL}' tidak ada, pencarian certificate hanya di Resolution notes.")

print("Jumlah baris :", len(df))
print("Kolom        :", list(df.columns))

## 7. Jalankan logic

In [ ]:
short_values = df[short_col] if short_col is not None else [""] * len(df)
resolved = [resolve_tags(r, s) for r, s in zip(df[res_col], short_values)]

df[TAGS_RAW_COL] = [raw for _, raw, _ in resolved]
df[TAGS_COL]     = [", ".join(tags) for tags, _, _ in resolved]
df[SOURCE_COL]   = [src for _, _, src in resolved]

df[NO_RISK_COL] = df[TAGS_COL].apply(
    lambda s: ", ".join(t for t in split_tag_string(s) if t.casefold() != HIGH_RISK.casefold())
)

pairs = df[TAG12_SOURCE_COL].apply(lambda s: pick_tag1_tag2(split_tag_string(s)))
df[TAG1_COL] = [p[0] for p in pairs]
df[TAG2_COL] = [p[1] for p in pairs]

print("Selesai:", len(df), "baris diproses")
print(df[SOURCE_COL].value_counts().to_string())

## 8. Lihat hasil & simpan ke Excel

In [ ]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

print(df[[TAGS_RAW_COL, TAGS_COL, SOURCE_COL, NO_RISK_COL, TAG1_COL, TAG2_COL]].to_string())

df.to_excel(OUTPUT_FILE, index=False)
print(f"\nTersimpan: {OUTPUT_FILE}")